# 2. Limpieza y normalización de lluvia diaria

Este notebook continúa el rol de ser **parte del proceso metodológico para la variable climática**. Su propósito es transformar el CSV consolidado de precipitaciones diarias en una serie utilizable, auditable y alineada con la misma ventana temporal del análisis operativo, sin introducir supuestos artificiales sobre días faltantes.

Conviene explicitar el origen del insumo: el CSV consumido aquí **no es una fuente nacida digitalmente**, sino un artefacto intermedio construido a partir de registros pluviométricos en papel que fueron preservados como imágenes y PDFs. En una etapa upstream previa, scripts de la familia `data/raw/rain/Registro Pluviometrico Montecarlo/` utilizaron la Google GenAI API (Gemini) para hacer OCR y extracción estructurada sobre documentos no tabulares, convirtiendo esas planillas escaneadas en filas diarias de precipitación.

Este notebook empieza **después** de esa extracción. No relee imágenes ni invoca la API: toma el CSV ya tabularizado y consolidado, limpia formatos, controla duplicados, identifica exclusiones, reconstruye el calendario `2021-01-01` → `2026-03-31` y deja `lluvia_diaria_clean.parquet` con marcas explícitas de observación real.

## Cómo leer este notebook

1. **Fuente y ventana temporal**: confirma qué archivo se usa y qué horizonte cubre.
2. **Resumen de normalización**: muestra cuántos registros se conservaron, cuántos se excluyeron y cómo queda cubierto el calendario ya reparado.
3. **Artifacts y vistas rápidas**: permite verificar que la salida final sea consistente antes de integrarla con reclamos.

> Alcance: depuración, normalización y auditabilidad de lluvia diaria sobre el CSV consolidado posterior al OCR/extracción estructurada.  
> Fuera de alcance: OCR, extracción multimodal desde imágenes/PDFs, inferencia causal sobre clima y demanda, o imputación agresiva de precipitaciones no observadas.


In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
from helpers.paths import require_input
from helpers.rainfall_cleaning import DATE_WINDOW_END, DATE_WINDOW_START, run_rainfall_pipeline


## 1. Fuente utilizada y ventana analítica

El primer control es deliberadamente simple: asegurar que la notebook parte del archivo correcto y que la serie se recorta con la misma ventana que usa el resto del trabajo. Sin esa alineación, la comparación entre lluvia y reclamos quedaría metodológicamente débil.

Metodológicamente, este bloque valida un **insumo derivado**: el archivo ya consolidado tras la extracción OCR/estructurada de los registros escaneados y la posterior recuperación manual de meses que originalmente habían quedado incompletos. Por eso la trazabilidad importa tanto como el contenido; lo que se verifica acá no es el documento papel original, sino el dataset tabular reparado que hereda esa etapa previa y queda listo para depuración reproducible.


In [2]:
rain_path = require_input('lluvias')
pd.DataFrame([{
    'dataset': 'lluvias',
    'path': str(rain_path.relative_to(ROOT)),
    'window_start': DATE_WINDOW_START,
    'window_end': DATE_WINDOW_END,
}])

,dataset,path,window_start,window_end
0,lluvias,data/raw/rain/dataset_lluvias_diario_con_obs.csv,2021-01-01,2026-03-31


## 2. Resumen de limpieza y cobertura del calendario

La tabla siguiente condensa la pregunta importante: **qué parte de la serie original quedó disponible para análisis y qué parte no**. En la versión actual ya no aparece el faltante histórico de 182 días, porque el insumo upstream fue completado antes de esta corrida. Por eso conviene leer la tabla distinguiendo entre filas crudas, exclusiones auditadas, días efectivamente observados y calendario total esperado: esa separación sigue haciendo visible la auditabilidad del dato climático, pero ahora sobre una serie reparada y completa dentro de la ventana.


In [3]:
result = run_rainfall_pipeline()
summary = pd.Series(result['summary'], name='value').rename_axis('metric').reset_index()
summary

,metric,value
0,rain_rows_raw,2012
1,rain_rows_excluded,96
2,rain_days_observed,1916
3,rain_days_calendar,1916
4,rain_days_missing_source,0
5,rain_days_duplicate_collapsed,0


## 3. Artifacts generados

Una vez terminada la depuración, el notebook persiste tanto la serie limpia como sus reportes auxiliares. Esta tabla deja trazabilidad de los archivos que luego se consumen en exploración, feature engineering y modelado.


In [4]:
pd.DataFrame({
    'artifact': list(result['artifacts'].keys()),
    'path': [str(Path(path).relative_to(ROOT)) for path in result['artifacts'].values()]
})

,artifact,path
0,lluvia_diaria_clean,data/processed/lluvia_diaria_clean.parquet
1,exclusiones_lluvia,data/intermediate/exclusiones_lluvia.parquet
2,lluvia_diaria_observada,data/intermediate/lluvia_diaria_observada.parquet
3,report,reports/phase3_rainfall_audit.md


## 4. Verificación rápida de la serie resultante

Antes de seguir conviene mirar dos cosas: la distribución de `lluvia_status`, que ahora debería quedar enteramente en `observed` tras la recuperación manual / completitud upstream, y una muestra de filas para verificar formato, calendario y variables derivadas. Es un control corto, pero clave para confirmar que la serie reparada ya no arrastra huecos de fuente dentro de la ventana analítica.


In [5]:
lluvia_clean = pd.read_parquet(result['artifacts']['lluvia_diaria_clean'])
lluvia_clean['lluvia_status'].value_counts(dropna=False).rename_axis('lluvia_status').reset_index(name='days')

,lluvia_status,days
0,observed,1916


In [6]:
lluvia_clean.head()

,fecha,lluvia_mm,observacion_codigo,source_row_count,source_file_count,lluvia_status
0,2021-01-01,0.0,A,1,1,observed
1,2021-01-02,0.0,A,1,1,observed
2,2021-01-03,0.0,A,1,1,observed
3,2021-01-04,5.5,A•,1,1,observed
4,2021-01-05,45.5,/B•,1,1,observed
